# Tokenization as a hidden variable in factual recall

Runs the full experiment on Colab. **Runtime → Change runtime type → T4 GPU** before starting.

Order: (1) setup, (2) smoke test on a tiny model (~2 min), (3) full runs (~45–70 min on a T4), (4) analysis, (5) download results.
Each model's results are saved as soon as it finishes, so if Colab disconnects just re-run the setup cells and step 3 — finished models are skipped.

## 1. Setup

In [ ]:
!nvidia-smi -L
!pip -q install datasets

In [ ]:
# Get the code: either clone your GitHub repo (replace the URL) or upload token-recall.zip
import os, zipfile
REPO_URL = ""  # e.g. "https://github.com/<your-username>/token-recall.git"
if not os.path.exists("token-recall/src"):
    if REPO_URL:
        !git clone -q $REPO_URL token-recall
    else:
        from google.colab import files
        up = files.upload()          # choose token-recall.zip
        zipfile.ZipFile(next(iter(up))).extractall(".")
%cd token-recall
!ls src

## 2. Smoke test (tiny model, 200 facts)
Check that the tokenisation mismatch rate is ~0 and the example prompt looks sensible.

In [ ]:
!python src/probe.py --model EleutherAI/pythia-70m-deduped --n 200 --out_dir results_smoke
!python src/analyze.py --results results_smoke --out smoke --min_n 5 > /dev/null && echo "analysis OK"

## 3. Full runs
Same tokenizer across all Pythia sizes, so only scale varies. Set `N = None` for all ~14k facts, or e.g. `N = 6000` if you are short on time.

In [ ]:
N = None
MODELS = [
    ("EleutherAI/pythia-160m-deduped", 32),
    ("EleutherAI/pythia-410m-deduped", 32),
    ("EleutherAI/pythia-1.4b-deduped", 32),
    ("EleutherAI/pythia-2.8b-deduped", 16),
]
n_arg = f"--n {N}" if N else ""
for name, bs in MODELS:
    out = f"results/{name.split('/')[-1]}.csv"
    if os.path.exists(out):
        print("skip (done):", out); continue
    !python src/probe.py --model $name --batch_size $bs $n_arg --out_dir results

## 4. Analysis → tables/*.tex, figures/*.pdf, results/summary.json

In [ ]:
!python src/analyze.py --results results --out .
from IPython.display import Image, display
for f in ["fig1_accuracy_by_tokens", "fig2_popularity_strata", "fig3_positional"]:
    display(Image(f"figures/{f}.png"))

In [ ]:
# Error analysis material: multi-token objects the largest model got wrong
import pandas as pd
d = pd.read_csv("results/pythia-2.8b-deduped.csv")
d[(d.n_obj_tok >= 3) & (~d.gen_correct)].sample(25, random_state=0)[["prop","subj","obj","n_obj_tok","gen","tf_vec"]]

## 5. Download everything (commit these to your Git repo too)

In [ ]:
!zip -qr outputs.zip results tables figures
from google.colab import files
files.download("outputs.zip")